# Embedding Model Fine-tuning with Sentence Transformers

[![Open In Colab](https://img.shields.io/badge/Open%20In-Colab-blue?style=for-the-badge&logo=google-colab)](https://colab.research.google.com/github/dnth/rag-datakit/blob/main/nbs/02_train.ipynb)
[![Open In Kaggle](https://img.shields.io/badge/Open%20In-Kaggle-blue?style=for-the-badge&logo=kaggle)](https://kaggle.com/kernels/welcome?src=https://github.com/dnth/rag-datakit/blob/main/nbs/02_train.ipynb)

This notebook demonstrates how to fine-tune an embedding model using the synthetic triplet data generated from Singapore SkillsFuture Framework job descriptions. We'll use the sentence-transformers library to train a model that can better understand job-related semantic similarity for improved retrieval and matching.

## What you'll learn:
- How to set up sentence-transformers training pipeline
- Configuring training arguments for embedding models
- Using MultipleNegativesRankingLoss for triplet training
- Monitoring training with Weights & Biases
- Saving and publishing trained models

## Installation

Install the rag-datakit package which includes all necessary dependencies including distilabel, transformers, and dataset utilities. Uncomment the cell below to install if you haven't already.

On Google Colab you might need to uninstall the existing packages due to conflicting versions.

In [3]:
!pip uninstall -y transformers torch torchvision

Found existing installation: transformers 4.56.1
Uninstalling transformers-4.56.1:
  Successfully uninstalled transformers-4.56.1
Found existing installation: torch 2.8.0
Uninstalling torch-2.8.0:
  Successfully uninstalled torch-2.8.0


In [4]:
!pip install git+https://github.com/dnth/rag-datakit.git

  Cloning https://github.com/dnth/rag-datakit.git to /tmp/pip-req-build-6hp5r6dc
  Running command git clone --filter=blob:none --quiet https://github.com/dnth/rag-datakit.git /tmp/pip-req-build-6hp5r6dc


  Resolved https://github.com/dnth/rag-datakit.git to commit dde15a4fa8c09263c0842386287d3ddab4754e6a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached torch-2.8.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached transformers-4.56.1-py3-none-any.whl.metadata (42 kB)
Using cached transformers-4.56.1-py3-none-any.whl (11.6 MB)
Using cached torch-2.8.0-cp312-cp312-manylinux_2_28_x86_64.whl (887.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers] [transformers]


## Import Required Libraries

We begin by importing the essential libraries for training our embedding model:

- **sentence_transformers**: The core library providing tools for training and evaluating sentence transformers
- **datasets**: Hugging Face's library for loading and managing our training dataset
- **wandb**: Weights & Biases for experiment tracking and visualization of training metrics

In [6]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import BatchSamplers
from datasets import load_dataset

## Dataset Loading and Inspection

We load the preprocessed training dataset with train/validation splits that were created in the previous notebook. This dataset contains triplets specifically formatted for embedding model training using contrastive learning approaches.

**Dataset source**: `dnth/ssf-train-valid`  
**Structure**:
- **anchor**: Original job descriptions from the SkillsFuture Framework
- **positive**: Semantically similar paraphrases generated through synthetic data techniques
- **negative**: Semantically different job descriptions used as contrastive examples

Let's examine the dataset structure and review a sample triplet to understand the data format:

In [7]:
dataset = load_dataset("Fatin757/ssf-train-valid_v3")
dataset

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 6032
    })
    valid: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 1508
    })
})

In [6]:
dataset['valid'][0]

{'anchor': "The Manager - Standards and Practices (S&P) ensures that content delivered by the organisation complies with the regulatory requirements and censorship norms of the local territories where the content may be available. He/She also provides advisory ratings for the content based on the regulatory guidelines. He keeps abreast of the local, cultural and political norms and sensitivities to support the creation of content classification guidelines. The work involves coordinating internal and external processes for delivery within tight timelines. He is highly accountable for the organisation's brand and reputation given the sensitivities of content classification. He should be comfortable coordinating with internal and external stakeholders in order to balance the organisation's priorities with compliance to guidelines and norms. He should be effective at planning and organising. He should also be aware of the regulatory, political and cultural landscape and possess a keen eye 

## Initialize Weights & Biases and Model Configuration

We initialize Weights & Biases for experiment tracking and define our base model and save path. We're using the `all-minilm-l6-v2` model as our starting point, which is a efficient, general-purpose sentence embedding model that provides a good balance between speed and performance.

In [8]:
import wandb

model_id = "answerdotai/ModernBERT-base"
save_model_path = "./models/answerdotai/ModernBERT-base"

# wandb.login()
wandb.init(project="rag-datakit-finetunes", name="answerdotai/ModernBERT-base")

## Configure Training Arguments

We configure the training arguments for our embedding model. These parameters control various aspects of the training process including batch sizes, learning rate, precision settings, and checkpointing behavior. Key configurations include:

- 5 training epochs with cosine learning rate scheduler
- Mixed precision training using bf16 for efficiency
- Gradient accumulation to achieve an effective batch size of 512
- NO_DUPLICATES batch sampler to ensure diverse negative samples
- Checkpointing at the end of each epoch with a limit of 3 saved models
- Evaluation after each epoch to monitor validation loss

In [10]:
args = SentenceTransformerTrainingArguments(
    output_dir=save_model_path,
    num_train_epochs=5,                         # number of epochs
    per_device_train_batch_size=32,             # train batch size
    per_device_eval_batch_size=16,              # evaluation batch size
    warmup_ratio=0.1,                           # warmup ratio
    learning_rate=5e-5,                         # learning rate, 2e-5 is a good value
    lr_scheduler_type="cosine",                 # use cosine learning rate scheduler
    optim="adamw_torch_fused",                  # use fused adamw optimizer
    #tf32=False,                                  # use tf32 precision
    bf16=True,                                  # use bf16 precision
    #fp16=True,                                   # use fp16 precision
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
    eval_strategy="epoch",                      # evaluate after each epoch
    save_strategy="epoch",                      # save after each epoch
    logging_strategy="epoch",                   # log after each epoch
    save_total_limit=3,                         # save only the last 3 models
    load_best_model_at_end=True,                # load the best model when training ends
    report_to="wandb",
  
 
    )

## Initialize Model, Loss Function, and Trainer

We initialize our sentence transformer model and configure the training components:

- Load the base `all-minilm-l6-v2` model from sentence-transformers
- Configure `MultipleNegativesRankingLoss` which is ideal for triplet training as it maximizes the similarity between anchor and positive pairs while minimizing similarity between anchor and negative pairs
- Set up the `SentenceTransformerTrainer` with our model, training arguments, datasets, and loss function

In [11]:
model = SentenceTransformer(model_id)
train_loss = MultipleNegativesRankingLoss(model)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['valid'],  
    loss=train_loss,
)

No sentence-transformers model found with name answerdotai/ModernBERT-base. Creating a new one with mean pooling.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

## Execute Model Training

We start the training process using our configured trainer. The model will train for 5 epochs, with evaluation happening after each epoch. The training progress and metrics are tracked through Weights & Biases, showing both training and validation loss metrics.

As training progresses, we can observe the validation loss decreasing, indicating that our model is learning to distinguish between semantically similar and dissimilar job descriptions.

In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.336200,0.009619
2,0.007900,0.005570
3,0.002900,0.004187
4,0.002600,0.003718
5,0.002300,0.003190


TrainOutput(global_step=945, training_loss=0.07039138715733927, metrics={'train_runtime': 381.4221, 'train_samples_per_second': 79.073, 'train_steps_per_second': 2.478, 'total_flos': 0.0, 'train_loss': 0.07039138715733927, 'epoch': 5.0})

## Save and Upload Trained Model

After training is complete, we save the fine-tuned model to disk and upload it to Weights & Biases for versioning and sharing. The saved model includes all necessary components:

- Model weights and configuration
- Tokenizer files
- Pooling layer configuration
- Normalization components

This ensures that the model can be easily loaded and used for inference later.

In [13]:
trainer.save_model()

In [14]:
import os
wandb.save(os.path.join(save_model_path, "*"))

wandb: WARNING Symlinked 14 files into the W&B run directory, call wandb.save again to sync new files.


['/root/rag-datakit/nbs/wandb/run-20250918_062320-g8kf7e3q/files/models/answerdotai/ModernBERT-base/config_sentence_transformers.json',
 '/root/rag-datakit/nbs/wandb/run-20250918_062320-g8kf7e3q/files/models/answerdotai/ModernBERT-base/1_Pooling',
 '/root/rag-datakit/nbs/wandb/run-20250918_062320-g8kf7e3q/files/models/answerdotai/ModernBERT-base/tokenizer_config.json',
 '/root/rag-datakit/nbs/wandb/run-20250918_062320-g8kf7e3q/files/models/answerdotai/ModernBERT-base/special_tokens_map.json',
 '/root/rag-datakit/nbs/wandb/run-20250918_062320-g8kf7e3q/files/models/answerdotai/ModernBERT-base/modules.json',
 '/root/rag-datakit/nbs/wandb/run-20250918_062320-g8kf7e3q/files/models/answerdotai/ModernBERT-base/README.md',
 '/root/rag-datakit/nbs/wandb/run-20250918_062320-g8kf7e3q/files/models/answerdotai/ModernBERT-base/config.json',
 '/root/rag-datakit/nbs/wandb/run-20250918_062320-g8kf7e3q/files/models/answerdotai/ModernBERT-base/checkpoint-756',
 '/root/rag-datakit/nbs/wandb/run-20250918_0

## Training Results and Next Steps

The training completed successfully with a final validation loss of 0.00813, showing that our model has learned to effectively distinguish between semantically similar and dissimilar job descriptions. The Weights & Biases dashboard provides detailed metrics and visualizations of the training process.

### Next Steps

To use this model in production:

1. **Load the model** using `SentenceTransformer('./models/all-minilm-l6-v2')`
2. **Evaluate** on a test set to verify performance on unseen data
3. **Deploy** in your RAG pipeline for improved job description matching
4. **Publish** to the Hugging Face Hub (uncomment the last cell) to share with the community

The fine-tuned model is now ready to provide more accurate semantic similarity scores for job descriptions in your retrieval-augmented generation workflows.

In [15]:
wandb.finish()

eval/loss,█▄▂▂▁
eval/runtime,█▂▂▁▃
eval/samples_per_second,▁▇▇█▅
eval/steps_per_second,▁▇▇█▅
train/epoch,▁▁▃▃▅▅▆▆███
train/global_step,▁▁▃▃▅▅▆▆███
train/grad_norm,▁▁▁▁█
train/learning_rate,█▆▄▂▁
train/loss,█▁▁▁▁
eval/loss,0.00319
eval/runtime,14.0325


In [18]:
trainer.model.push_to_hub("Fatin757/ssf-retriever-modernbert-v2", exist_ok=True)

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmpp0vqiybr/model.safetensors    :   0%|          |  552kB /  596MB            

'https://huggingface.co/Fatin757/ssf-retriever-modernbert-v2/commit/9c347e7836c8eacac05e6352b68595ae908b3f9e'